# Notebook 03 — Document Parsing & Chunking

**User-owned notebook for RAG exploration.**

**What to explore here:**
- Parse PDF (page-level text extraction)
- Parse DOCX (paragraph-level)
- Parse Markdown / plain text
- Inspect `ParsedDocument` structure
- Run through chunker — inspect `Chunk` objects
- Verify page_number and para_number tracking
- Test edge cases (empty pages, short docs)

**Bring your own documents** — place PDFs/DOCX/TXT files in `../data/uploads/` and update paths below.

In [7]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from backend.ingestion.parsers.pdf_parser import PDFParser
from backend.ingestion.parsers.docx_parser import DOCXParser
from backend.ingestion.parsers.text_parser import TextParser
from backend.ingestion.chunker import DocumentChunker
from backend.ingestion.pipeline import parse_document, chunk_document, get_parser
import uuid

print('Imports OK')

Imports OK


## 1. Parse a PDF

In [ ]:
PDF_PATH = '../data/uploads/sample.pdf'

if Path(PDF_PATH).exists():
    parsed_pdf = parse_document(PDF_PATH, title='Sample Policy')
    print(f'Title: {parsed_pdf.title}')
    print(f'Pages: {parsed_pdf.page_count}')
    print(f'\nPage 1 preview (first 300 chars):')
    print(parsed_pdf.pages[0].text[:300] if parsed_pdf.pages else '(empty)')
    print(f'\nParagraphs on page 1: {len(parsed_pdf.pages[0].paragraphs)}')
else:
    print(f'No PDF at {PDF_PATH} — place a PDF there to test.')
    print('Creating a synthetic text doc for demonstration...')
    # Create a minimal sample text file for testing
    sample_text = ''
    for i in range(5):
        sample_text += f'Section {i+1}\n\n'
        sample_text += f'This is paragraph {i+1}a of the IT security policy. It describes access control requirements for all employees.\n\n'
        sample_text += f'This is paragraph {i+1}b. Users must authenticate using MFA before accessing sensitive systems. All access is logged.\n\n'
    Path('../data/uploads/sample.txt').write_text(sample_text)
    print('Created ../data/uploads/sample.txt')

No PDF at ../data/uploads/sample.pdf — place a PDF there to test.
Creating a synthetic text doc for demonstration...
Created ../data/uploads/sample.txt


## 2. Parse a Text / Markdown File

In [3]:
TXT_PATH = '../data/uploads/sample.txt'

if Path(TXT_PATH).exists():
    parsed_txt = parse_document(TXT_PATH, title='Sample Text Policy')
    print(f'Title: {parsed_txt.title}')
    print(f'Logical pages: {parsed_txt.page_count}')
    for page in parsed_txt.pages:
        print(f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, {len(page.text)} chars')
else:
    print(f'File not found: {TXT_PATH}')

Title: Sample Text Policy
Logical pages: 1
  Page 1: 15 paragraphs, 1173 chars


## 3. Run Through Chunker

In [11]:
# Use whichever document parsed successfully above
doc_to_chunk = None
for path in [PDF_PATH, TXT_PATH]:
    if Path(path).exists():
        doc_to_chunk = parse_document(path)
        break

if doc_to_chunk:
    fake_doc_id = str(uuid.uuid4())
    fake_roles = ['role-uuid-1', 'role-uuid-2']  # Replace with real role UUIDs
    
    chunks = chunk_document(
        doc_to_chunk,
        doc_id=fake_doc_id,
        allowed_roles=fake_roles,
        is_archived=False,
        chunk_size=500,
        chunk_overlap=50,
    )
    
    print(f'Total chunks: {len(chunks)}')
    print(f'\nFirst chunk:')
    c = chunks[0]
    print(f'  chunk_id: {c.chunk_id}')
    print(f'  doc_id: {c.doc_id}')
    print(f'  page_number: {c.page_number}')
    print(f'  para_number: {c.para_number}')
    print(f'  is_archived: {c.is_archived}')
    print(f'  allowed_roles: {c.allowed_roles}')
    print(f'  text ({len(c.text)} chars): {c.text[:200]}...')

Total chunks: 13

First chunk:
  chunk_id: 9424fea686e376cc
  doc_id: 70765e93-a317-42ec-aa50-e34dd954b7f4
  page_number: 1
  para_number: 0
  is_archived: False
  allowed_roles: ['role-uuid-1', 'role-uuid-2']
  text (497 chars): Firewall Management Policy
Hills Shire Care Private Hospital
Document Title: Firewall Management Policy
Document Number: HSC-FMP-001
Version Number: 1.0
Effective Date: July 1, 2024
Next Review Date: ...


## 4. Inspect Chunk Distribution

In [12]:
if doc_to_chunk and chunks:
    import collections
    
    page_dist = collections.Counter(c.page_number for c in chunks)
    print('Chunks per page:')
    for page, count in sorted(page_dist.items()):
        print(f'  Page {page}: {count} chunks')
    
    text_lens = [len(c.text) for c in chunks]
    print(f'\nChunk text length stats:')
    print(f'  Min: {min(text_lens)}')
    print(f'  Max: {max(text_lens)}')
    print(f'  Avg: {sum(text_lens)/len(text_lens):.0f}')

Chunks per page:
  Page 1: 6 chunks
  Page 2: 7 chunks

Chunk text length stats:
  Min: 153
  Max: 497
  Avg: 421


## 5. Edge Cases

In [6]:
from backend.ingestion.parsers.base import ParsedDocument, ParsedPage

# Empty document
empty_doc = ParsedDocument(source_path='empty.txt', title='Empty', pages=[])
empty_chunks = chunk_document(empty_doc, 'test-id', ['role-1'])
assert empty_chunks == [], f'Expected empty chunks, got {empty_chunks}'
print('Empty document → 0 chunks: PASS')

# Single page with minimal content
tiny_doc = ParsedDocument(
    source_path='tiny.txt',
    title='Tiny',
    pages=[ParsedPage(page_number=1, text='Short policy text.', paragraphs=['Short policy text.'])]
)
tiny_chunks = chunk_document(tiny_doc, 'tiny-id', ['role-1'])
assert len(tiny_chunks) >= 1
print(f'Single short page → {len(tiny_chunks)} chunk(s): PASS')

# Chunk IDs are stable (same input → same chunk_id)
chunks_a = chunk_document(tiny_doc, 'stable-id', ['role-1'])
chunks_b = chunk_document(tiny_doc, 'stable-id', ['role-1'])
assert chunks_a[0].chunk_id == chunks_b[0].chunk_id
print('Chunk IDs are deterministic (stable for re-ingestion): PASS')

Empty document → 0 chunks: PASS
Single short page → 1 chunk(s): PASS
Chunk IDs are deterministic (stable for re-ingestion): PASS


## Summary

Parser + chunker layer is working:
- PDF → page-level text extraction ✓
- DOCX → paragraph-level grouping into logical pages ✓
- Text/MD → blank-line paragraph splitting ✓
- Chunker produces Chunk objects with page/para tracking ✓
- chunk_id is deterministic for idempotent re-ingestion ✓

**Next:** Notebook 04 — RAG Pipeline (connect ChromaDB + Ollama)